In [24]:
# Load the Drive helper and mount
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [25]:
!rm -rf colab_llm_utils

In [26]:
!git clone -b multiaxial https://github.com/ravy101/colab_llm_utils.git

Cloning into 'colab_llm_utils'...
remote: Enumerating objects: 1561, done.
remote: Counting objects: 100% (83/83), done.
remote: Compressing objects: 100% (59/59), done.
remote: Total 1561 (delta 49), reused 55 (delta 24), pack-reused 1478 (from 2)
Receiving objects: 100% (1561/1561), 1.05 MiB | 13.73 MiB/s, done.
Resolving deltas: 100% (994/994), done.


In [27]:
!pip install openai pandas evaluate tqdm -q

In [28]:
import colab_llm_utils
from colab_llm_utils import configs

In [29]:
import pandas as pd
import os
import json
import time
from openai import OpenAI, OpenAIError
from google.colab import userdata
from tqdm.notebook import tqdm

In [30]:

try:
    api_key = userdata.get('OPENAI_TOKEN')
    client = OpenAI(api_key=api_key)
    print("✅ API Key loaded successfully.")
except Exception as e:
    print("❌ Error loading API key. Did you add 'OPENAI_API_KEY' to Colab Secrets?")


✅ API Key loaded successfully.


In [77]:
dataset_config = configs.datasets.routerbench_rag
model_config = configs.models.qwen3_8b

short_name = model_config['model_name'].split('/')[-1]

FEW_SHOT = True
P_TRUE = False
FACTUALITY_ONLY = True
COMPARISON = False

#special_tag = ""
#special_tag = "k_shot"
#special_tag="rag"
special_tag = "verify"

drive_path = f"/content/drive/MyDrive/phase3/Llama/{dataset_config['clean_name']}/"

In [78]:
df_res = pd.read_pickle(os.path.join(drive_path, f"{dataset_config['dataset_name']}{short_name}{special_tag}.pickle"))

In [79]:
dataset_config

{'clean_name': 'RouterBench-RAG',
 'dataset_name': 'routerbench_rag',
 'dataset_location': 'local_pickle',
 'local_file': '/content/drive/MyDrive/phase3/dataset/routerbench_rag_out.pkl',
 'local_test': '/content/drive/MyDrive/phase3/dataset/routerbench_2k_sample.pkl',
 'options': ['A', 'B', 'C', 'D', 'E'],
 'subset': 'train',
 'task_type': 'qa',
 'dict_ans': False,
 'shuffle': True,
 'doc_to_text': <function colab_llm_utils.configs.datasets.doc_to_text_routerbench(item)>,
 'doc_to_ans': <function colab_llm_utils.configs.datasets.doc_to_answer_routerbench(item)>,
 'doc_to_choices': <function colab_llm_utils.configs.datasets.doc_to_choices_routerbench(item)>}

In [80]:
df_res.head()

,prompts,responses,ans,logit_outs,token_outs,blocks,meta,task_family,source_row,likes,all_probas,top_probas,chow_av,chow_sum,chow_quantile,log_chow_av,thinking_text,scored_responses,em,is_correct
0,Review the candidate answer below.\n\nQUESTION...,[The 1996 Nobel Prize in Chemistry was awarded...,"Harry Kroto, Richard Smalley, and Robert Curl","[{785: 45.0, 41215: 28.54166603088379, 576: 28...","[151644, 872, 198, 19432, 279, 9144, 4226, 368...",{},{},qa,"{'youcom/rag|total_cost': 0.0050443, 'youcom/r...","[45.0, 53.75, 49.58333206176758, 48.125, 48.12...","[[0.9999998106107345, 7.116005804119964e-08, 6...","[0.9999998106107345, 0.999999981884174, 0.9999...",0.999068,48.954310,1.000000,-0.000939,None,[The 1996 Nobel Prize in Chemistry was awarded...,0.375000,0.375000
1,Review the candidate answer below.\n\nQUESTION...,[The answer should be: \n2021],Blockchains attained a 13.5% adoption rate wit...,"[{785: 36.041664123535156, 17: 36.0, 11190: 24...","[151644, 872, 198, 19432, 279, 9144, 4226, 368...",{},{},qa,"{'youcom/rag|total_cost': 0.00501505, 'youcom/...","[36.041664123535156, 45.416664123535156, 45.62...","[[0.5104088377396656, 0.4895800210400113, 4.37...","[0.5104088377396656, 0.9811828081345829, 0.999...",0.929039,10.219429,0.983562,-0.089010,None,[The answer should be: \n2021],0.000000,0.000000
2,Review the candidate answer below.\n\nQUESTION...,[The U.S. state that became the first to ban t...,Missouri became the first U.S. state to ban th...,"[{785: 33.541664123535156, 77469: 29.5, 3564: ...","[151644, 872, 198, 19432, 279, 9144, 4226, 368...",{},{},qa,"{'youcom/rag|total_cost': 0.0050378, 'youcom/r...","[33.541664123535156, 42.70833206176758, 41.25,...","[[0.9686465282647142, 0.017017387949570806, 0....","[0.9686465282647142, 0.991168833523462, 0.9999...",0.980210,40.188611,0.999999,-0.024275,None,[The U.S. state that became the first to ban t...,0.647059,0.647059
3,Review the candidate answer below.\n\nQUESTION...,[Georgi Sudakov scored against **Bayer Leverku...,Barcelona,"[{9499: 45.83333206176758, 785: 34.79166412353...","[151644, 872, 198, 19432, 279, 9144, 4226, 368...",{},{},qa,"{'youcom/rag|total_cost': 0.00498645, 'youcom/...","[45.83333206176758, 39.791664123535156, 49.375...","[[0.9999771963612064, 1.6019709627200904e-05, ...","[0.9999771963612064, 0.9999936290267137, 0.999...",0.994908,53.725033,1.000000,-0.005296,None,[Georgi Sudakov scored against **Bayer Leverku...,0.000000,0.000000
4,Review the candidate answer below.\n\nQUESTION...,[Cryptocurrency transactions are **recorded an...,Cryptocurrency transactions are recorded as bl...,"[{53367: 42.08333206176758, 785: 34.7916641235...","[151644, 872, 198, 19432, 279, 9144, 4226, 368...",{},{},qa,"{'youcom/rag|total_cost': 0.00514505, 'youcom/...","[42.08333206176758, 42.08333206176758, 52.2916...","[[0.9987220659061234, 0.0006803204032056014, 0...","[0.9987220659061234, 0.9999726049922312, 0.999...",0.931418,119.221495,0.999994,-0.089550,None,[Cryptocurrency transactions are **recorded an...,0.205128,0.205128


In [81]:
if special_tag == 'verify':
  df_res['verify_prompt'] = df_res['prompts'].copy()
  df_res['prompts'] = [p.split("\n\n")[1] for p in df_res['prompts'][:2000]]


In [82]:

results = []
save_frequency = 100 # Save to CSV every 100 rows
output_filename = "scored_results_partial.csv"

print(f"Starting scoring for {len(df_res)} items...")

# Iterate through the DataFrame with a progress bar
for index, row in tqdm(df_res.iterrows(), total=len(df_res)):

    # Run the scorer
    response = row['responses']
    if isinstance(response, list):
        response = response[0]

    if dataset_config['task_type'] == 'qa' or dataset_config['task_type'] == 'mixed':
        if FEW_SHOT:
            prompt = row['prompts'].split("Provide a short answer without explanation")[-1]
            prompt = prompt.split("\nTranslation:")[0]
        else:
          prompt = row['prompts']
        result = colab_llm_utils.api_scorer.score_qa_pair(client, row['prompts'], response, row['ans'], api_model = "gpt-5-mini")
        # Save data
        results.append({
            'index': index,
            'question': row['prompts'],
            'response': response,
            'gpt_score': result.get('score'),
        })
    elif dataset_config['task_type'] == 'translation':
        if FEW_SHOT:
            prompt = row['prompts'].split("Translate  the following from ")[-1]
            prompt = prompt.split("\nTranslation:")[0]
        else:
          prompt = row['prompts']
        result = colab_llm_utils.api_scorer.score_translation(client, prompt, response, row['ans'])
        results.append({
            'index': index,
            'question': row['prompts'],
            'gpt_score': result.get('binary_adequacy'),
            'gpt_score_cont': result.get('adequacy'),
        })
    elif dataset_config['task_type'] == 'summarization':
        if COMPARISON:
            response2 = row['responses_large']
            if isinstance(response2, list):
                response2 = response2[0]
            prompt = row['prompts'].split(dataset_config['doc_type']+":")[-1]
            prompt = prompt.split("\n\nWrite a")[0]
            result = colab_llm_utils.api_scorer.compare_short_summary(client, prompt, response, response2, limit=dataset_config['limit'], crop=dataset_config['do_crop'], doc_type=dataset_config['doc_type'])
            results.append({
              'index': index,
              'question': prompt,
              'gpt_score': result.get('prefer_1'),
              'gpt_score_cont': result.get('prefer_2'),
          })
        else:
          if FEW_SHOT:
              prompt = row['prompts'].split(dataset_config['doc_type']+":")[-1]
              prompt = prompt.split("\n\nWrite a")[0]
          else:
            prompt = row['prompts']
          if dataset_config['do_crop']:
            result = colab_llm_utils.api_scorer.score_short_summary(client, prompt, response, limit=dataset_config['limit'], crop=dataset_config['do_crop'], doc_type=dataset_config['doc_type'])
          else:
            result = colab_llm_utils.api_scorer.score_summary(client, prompt, response)
          results.append({
              'index': index,
              'question': prompt,
              'gpt_score': result.get('faithful') and result.get('complete'),
              'gpt_score_cont': result.get('score'),
          })

    # Optional: Small sleep to be nice to the rate limits (though 4o-mini is fast)
    time.sleep(0.1)

    # Intermediate Save (so you don't lose progress)

    if (index + 1) % save_frequency == 0:
        int_df = pd.DataFrame(results)
        print(f"Mean score {int_df['gpt_score'].mean()}")
        int_df.to_csv(output_filename, index=False)



Starting scoring for 650 items...


  0%|          | 0/650 [00:00<?, ?it/s]

Mean score 0.37
Mean score 0.36
Mean score 0.3233333333333333
Mean score 0.3275
Mean score 0.332
Mean score 0.32666666666666666


In [83]:
result

{'score': 1}

In [84]:
# Final Save
final_df = pd.DataFrame(results)
df_res['gpt_score'] = final_df['gpt_score']
#df_res['gpt_score_alt'] = final_df['gpt_score_alt']
if dataset_config['task_type'] == 'translation' or dataset_config['task_type'] == "summarization":
  df_res['gpt_score_cont'] = final_df['gpt_score_cont']

In [85]:
final_df['gpt_score'].mean()

np.float64(0.32)

In [86]:
final_df.to_csv(os.path.join(drive_path, f"{dataset_config['dataset_name']}{short_name}{special_tag}scored_results_final.csv"), index=False)

print("✅ Scoring Complete!")
# Display first few rows
final_df.head()

✅ Scoring Complete!


,index,question,response,gpt_score
0,0,QUESTION:\nWho won the 1996 Nobel Prize in Che...,The 1996 Nobel Prize in Chemistry was awarded ...,1
1,1,QUESTION:\nBy what year did the use and develo...,The answer should be: \n2021,0
2,2,QUESTION:\nWhich U.S. state became the first t...,The U.S. state that became the first to ban th...,0
3,3,QUESTION:\nAgainst which team did Georgi Sudak...,Georgi Sudakov scored against **Bayer Leverkus...,0
4,4,QUESTION:\nHow are cryptocurrency transactions...,Cryptocurrency transactions are **recorded and...,1


In [87]:
df_res.to_pickle(os.path.join(drive_path, f"{dataset_config['dataset_name']}{short_name}{special_tag}.pickle"))

In [88]:
pd.read_csv(os.path.join(drive_path, f"{dataset_config['dataset_name']}{short_name}{special_tag}scored_results_final.csv"))


,index,question,response,gpt_score
0,0,QUESTION:\nWho won the 1996 Nobel Prize in Che...,The 1996 Nobel Prize in Chemistry was awarded ...,1
1,1,QUESTION:\nBy what year did the use and develo...,The answer should be: \n2021,0
2,2,QUESTION:\nWhich U.S. state became the first t...,The U.S. state that became the first to ban th...,0
3,3,QUESTION:\nAgainst which team did Georgi Sudak...,Georgi Sudakov scored against **Bayer Leverkus...,0
4,4,QUESTION:\nHow are cryptocurrency transactions...,Cryptocurrency transactions are **recorded and...,1
...,...,...,...,...
645,645,QUESTION:\nWhich species uses bioluminescence ...,The answer should be: \nThe species you're re...,0
646,646,QUESTION:\nWhen did Argentina announce the imp...,Argentina announced the implementation of a **...,0
647,647,QUESTION:\nAre the STALKER series games recomm...,The answer should be:\nBased on the informatio...,0
648,648,QUESTION:\nWho is the Federal Reserve Board Ch...,The Federal Reserve Board Chairman who conduct...,1


In [89]:
crash

NameError: name 'crash' is not defined

In [ ]:
special_tag = ""
STACK_MIX = True
if STACK_MIX:
  dataset = colab_llm_utils.configs.datasets.mixeval_ff
  dataset2 = colab_llm_utils.configs.datasets.mixeval_mc
  df_res1 = pd.read_pickle(os.path.join(drive_path, f"{dataset['dataset_name']}{short_name}{special_tag}.pickle"))
  df_res2 = pd.read_pickle(os.path.join(f"/content/drive/MyDrive/phase3/Llama/{dataset2['clean_name']}/", f"{dataset2['dataset_name']}{short_name}{special_tag}.pickle"))
  df_res2['gpt_score'] = (df_res2.em > .1).astype(int)
  pd.concat([df_res1, df_res2]).to_pickle(os.path.join(drive_path, f"{dataset['dataset_name']}{short_name}{special_tag}_stack.pickle"))

In [ ]:
df_res2.columns

In [ ]:
pd.concat([df_res1, df_res2])['gpt_score'].mean()

In [ ]:
df_res1.gpt_score.mean()

In [ ]:
 pd.concat([df_res1, df_res2]).shape